# 선형 회귀로 붓꽃의 꽃받침 길이 예측하기 (교과서 86~87쪽)

61쪽 (다) 문제: 일련번호 151인 붓꽃(꽃받침 너비 3.1, 꽃잎 길이 1.5, 꽃잎 너비 0.2, 종류 세토사)은 **꽃받침 길이**를 몰라요. 선형 회귀 모델을 만들어 꽃받침 길이를 예측해 봐요.

- 위에서부터 셀을 하나씩 실행해요. (셀 왼쪽 ▶ 버튼 또는 **Shift + Enter**)
- **중간에 오류가 나는 셀이 있어요.** 놀라지 말고 오류 메시지의 마지막 줄을 읽은 뒤, 그다음 셀로 해결해요.

## 0 데이터 불러오기
교과서의 `Iris1.csv`를 인터넷 주소에서 바로 불러와요. (파일을 올리지 않아도 돼요.)

In [ ]:
import pandas as pd  # 판다스 라이브러리 불러오기
df1 = pd.read_csv('https://raw.githubusercontent.com/sweet1203/ds_ai_class/main/lessons/u2-regression/assets/Iris1.csv', encoding = 'cp949')  # 한글이 포함된 CSV 파일을 데이터프레임으로 불러오기
df1.head()  # 앞 5개 데이터 출력

## 1 선형 회귀 모델 생성
### ① 독립 변수와 종속 변수를 설정한 후, 훈련 데이터와 테스트 데이터를 8 : 2로 나누기
꽃받침 너비, 꽃잎 길이, 꽃잎 너비, 종류는 독립 변수 X로, 꽃받침 길이는 종속 변수 y로 해요.

In [ ]:
X = df1[['꽃받침 너비', '꽃잎 길이', '꽃잎 너비', '종류']]  # 독립 변수
y = df1['꽃받침 길이']  # 종속 변수

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2,
                                                    shuffle = True, random_state = 7)  # shuffle = True: 데이터를 무작위로 섞음

### ② 선형 회귀 모델을 만들고 학습시키기
연속된 값을 예측하려고 `LinearRegression()`으로 회귀 모델을 만들고 학습시켜요.

⚠️ **실행하면 오류가 나요!** 출력 맨 아래 줄의 오류 메시지를 읽어 보세요.

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()  # 선형 회귀 모델 생성
model.fit(X_train, y_train)  # 선형 회귀 모델 학습

### 해석
`ValueError: could not convert string to float: '버지니카'`
→ 문자열 '버지니카'를 실수(float)로 바꿀 수 없다는 뜻이에요. **LinearRegression을 위한 독립 변수에는 범주형 데이터를 사용할 수 없기 때문**이에요.
그래서 범주형 데이터(종류)를 수치형 데이터로 바꾸는 **원-핫 인코딩(one-hot encoding)** 을 해요.

### ③ 원-핫 인코딩: 범주형 데이터를 수치형 데이터로 바꾸기
`get_dummies()`가 '종류' 속성을 '종류_버시컬러', '종류_버지니카', '종류_세토사' 세 속성으로 늘리고 값을 0이나 1로 채워요.

In [ ]:
df1 = pd.get_dummies(df1, columns = ['종류'], dtype = int)  # dtype = int: 0과 1로 표시 (최신 판다스는 넣지 않으면 True/False로 보여 줘요)
X = df1[['꽃받침 너비', '꽃잎 길이', '꽃잎 너비', '종류_버시컬러', '종류_버지니카', '종류_세토사']]
y = df1['꽃받침 길이']
print(X)

### ①, ② 다시 실행하기
교과서는 ③ 코드 실행 후 ①, ② 코드를 다시 실행하라고 해요. 그런데 위의 ① 셀은 이제 없어진 '종류' 열을 찾으므로 그대로 다시 실행하면 다른 오류가 나요.
그래서 **데이터 나누기와 모델 학습**만 아래 셀에 다시 적어 두었어요. 이번에는 오류 없이 학습돼요.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2,
                                                    shuffle = True, random_state = 7)  # 원-핫 인코딩한 X로 다시 나누기
model = LinearRegression()  # 선형 회귀 모델 생성
model.fit(X_train, y_train)  # 선형 회귀 모델 학습

### ④ 선형 회귀 모델의 기울기와 절편 찾기

In [ ]:
print('선형 회귀 가중치: {}'.format(model.coef_))  # coef_: 가중치(기울기)
print('선형 회귀 편향: {}'.format(model.intercept_))  # intercept_: 편향(절편)

## 2 성능 평가
- **평균제곱오차(MSE)**: 실젯값과 예측값 사이의 오차를 제곱하여 평균을 구한 것. **0에 가까울수록** 성능이 우수해요.
- **R²(결정 계수)**: 독립 변수가 종속 변수를 얼마만큼 설명해 주는지를 나타내는 비율. **1에 가까울수록** 성능이 우수해요.

먼저 교과서처럼 R²로 평가해요.

In [ ]:
from sklearn.metrics import r2_score  # 회귀 모델 성능 지표 모듈
y_pred = model.predict(X_test)  # 테스트 데이터에 대한 예측값
r2 = r2_score(y_test, y_pred)  # 결정 계수
print('결정 계수 R2: ', r2)

교과서 설명에 나온 평균제곱오차(MSE)도 계산해 봐요. (교과서 코드에 더한 셀)

In [ ]:
from sklearn.metrics import mean_squared_error  # 평균제곱오차 모듈
mse = mean_squared_error(y_test, y_pred)  # 평균제곱오차
print('평균제곱오차 MSE: ', mse)

## 3 꽃받침 길이 속성 예측
꽃받침 너비 3.1, 꽃잎 길이 1.5, 꽃잎 너비 0.2, 종류가 세토사인 붓꽃의 꽃받침 길이를 예측해요.
종류는 원-핫 인코딩한 순서(버시컬러, 버지니카, 세토사)대로 `0, 0, 1`로 넣어요.

In [ ]:
new_X = [[3.1, 1.5, 0.2, 0, 0, 1]]
predict = model.predict(new_X); print(predict)  # 경고(UserWarning) 문구가 함께 나와도 괜찮아요

### 더 해 보기
- ④에서 구한 가중치와 편향으로 회귀식 y = w1x1 + w2x2 + … + w6x6 + b 를 써 보세요.
- `new_X`의 종류를 버지니카 `0, 1, 0`으로 바꾸면 예측값이 어떻게 달라지나요?